# Sign Calculator

In [ ]:
from time import sleep
from pynq.overlays.base import BaseOverlay
 
# Convert 2-bit switch to signed value
def get_value(sw0, sw1):
    val = (sw1 << 1) | sw0
    if val == 0:
        return 0
    elif val == 1:
        return 1
    elif val == 2:
        return -2
    elif val == 3:
        return -1
 
# Display result
def display(base, result):
    if result < -7:
        result = -7
    if result > 8:
        result = 8
 
    # sign
    if result < 0:
        base.leds[3].on()
        result = abs(result)
    else:
        base.leds[3].off()
 
    # magnitude
    for i in range(3):
        if result & (1 << i):
            base.leds[i].on()
        else:
            base.leds[i].off()
 
def main():
    base = BaseOverlay('base.bit')
 
    mode = 0   # 0 = A, 1 = B
    operation = 0
    A = 0
    B = 0
 
    # Previous button states
    prev = [0, 0, 0, 0]
 
    print("Event-based Calculator Started")
 
    while True:
        # Current button states
        curr = [base.buttons[i].read() for i in range(4)]
 
        # BTN0 → Change Mode
        if curr[0] == 1 and prev[0] == 0:
            mode = (mode + 1) % 2
            print(f"Mode changed to {'A' if mode == 0 else 'B'}")
 
        # BTN1 → Capture value
        if curr[1] == 1 and prev[1] == 0:
            sw0 = base.switches[0].read()
            sw1 = base.switches[1].read()
            value = get_value(sw0, sw1)
 
            if mode == 0:
                A = value
                print(f"Captured A = {A}")
            else:
                B = value
                print(f"Captured B = {B}")
 
        # BTN2 → Change operation
        if curr[2] == 1 and prev[2] == 0:
            operation = (operation + 1) % 4
            ops = ["ADD", "SUB", "MUL", "DIV"]
            print(f"Operation = {ops[operation]}")
 
        # BTN3 → Compute
        if curr[3] == 1 and prev[3] == 0:
            if operation == 0:
                result = A + B
                op = "+"
            elif operation == 1:
                result = A - B
                op = "-"
            elif operation == 2:
                result = A * B
                op = "*"
            elif operation == 3:
                if B != 0:
                    result = int(A / B)
                else:
                    print("Divide by zero!")
                    result = 0
                op = "/"
 
            print(f"{A} {op} {B} = {result}")
            display(base, result)
 
        # Save previous state
        prev = curr.copy()
 
        sleep(0.05)  # small debounce
 
print("Code starting")
main()